In [3]:
from library import *
from skimage import color
from skimage.util import img_as_ubyte
from skimage.exposure import equalize_hist
from scipy.ndimage import center_of_mass
recording = '6-13-24'
VID_DIR = './pumpkin_videos/' + recording
videos = [
          'eat2_fs_00001', 'eat2_fs_00002', 'eat2_fs_00003', 'eat2_fs_00004', 'eat2_fs_00005',
          'eat2_ss_00001', 'eat2_ss_00002', 'eat2_ss_00003', 'eat2_ss_00004', 'eat2_ss_00005',
          'eat2_ff_00001', 'eat2_ff_00002', 'eat2_ff_00003', 'eat2_ff_00004', 'eat2_ff_00005',
          'eat2_sf_00001', 'eat2_sf_00002', 'eat2_sf_00003', 'eat2_sf_00004', 'eat2_sf_00005',
          'n2_fs_00001',   'n2_fs_00002',   'n2_fs_00003',   'n2_fs_00004',   'n2_fs_00005',
          'n2_ss_00001',   'n2_ss_00002',   'n2_ss_00003',   'n2_ss_00004',   'n2_ss_00005',
          'n2_ff_00001',   'n2_ff_00002',   'n2_ff_00003',   'n2_ff_00004',   'n2_ff_00005',
          'n2_sf_00001',   'n2_sf_00002',   'n2_sf_00003',   'n2_sf_00004',   'n2_sf_00005',
]

In [4]:
### Calculate COMs for all videos in recording
FILTER = True
for video in videos:
    print('---')
    print('Processing ' + video + '.wmv...')
    ### Load data
    fullName    = video + '_cropped.wmv'
    vidPath     = VID_DIR + '/cropped/' + fullName

    grinderName = VID_DIR + '/outputs/grinder_ROI/' + video + '_cropped_boxes.csv'
    grinders    = pd.read_csv(grinderName)
    grinders    = grinders.to_numpy()

    ### Load video
    vid        = cv2.VideoCapture(vidPath)
    tot_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))
    
    ### Main loop for calculating and saving grinder CoM
    coms = np.zeros((tot_frames,2))
    success = True
    i = 0
    while success and i < tot_frames-1:
        success, frame = vid.read()

        if success and np.sum(grinders[i]) > 0:
            # Calculate centroid of grinder in bounding box
            cropped = frame[int(grinders[i][1]): int(grinders[i][3]),int(grinders[i][0]): int(grinders[i][2])]
            box = color.rgb2gray(cropped) # grayscale
            box = img_as_ubyte(box)
            box = equalize_hist(box)  # equalize (increase grinder contrast)
            invert_box = 1 - box      # invert
            coms[i] = center_of_mass(invert_box)

            # Convert coordinates back to 250x250
            coms[i][0] += grinders[i][0]
            coms[i][1] += grinders[i][1]

        # Update counter
        i += 1
        
    # (OPTIONAL) Filter CoMs to fill gaps from dropped frames
    if FILTER:
        filtered_coms, _ = filter_ROIs(coms)
    
        # Save CoMs
        save_path = VID_DIR + '/outputs/COMs/' + video + '_coms.csv'
        np.savetxt(save_path,filtered_coms)
        print('Done. Saved to ' + save_path)
    
    else:
        # Save CoMs
        save_path = VID_DIR + '/outputs/COMs/' + video + '_coms.csv'
        np.savetxt(save_path,coms)
        print('Done. Saved to ' + save_path)

print('---')

---
Processing eat2_fs_00001.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_fs_00001_coms.csv
---
Processing eat2_fs_00002.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_fs_00002_coms.csv
---
Processing eat2_fs_00003.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_fs_00003_coms.csv
---
Processing eat2_fs_00004.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_fs_00004_coms.csv
---
Processing eat2_fs_00005.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_fs_00005_coms.csv
---
Processing eat2_ss_00001.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_ss_00001_coms.csv
---
Processing eat2_ss_00002.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_ss_00002_coms.csv
---
Processing eat2_ss_00003.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_ss_00003_coms.csv
---
Processing eat2_ss_00004.wmv...
Done. Saved to ./pumpkin_videos/6-13-24/outputs/COMs/eat2_ss_00004_c